# VELNOR General Intelligence — Qwen3-8B QLoRA (Free GPU)

This is the preferred advanced training path for VELNOR. It fine-tunes **Qwen3-8B** using 4-bit NF4 QLoRA and the expanded general-purpose curriculum from this branch: 288 training examples, 30 held-out examples, and a 27-case evaluation benchmark.

## Free GPU requirements
- Use a free CUDA runtime only. No paid GPU or cloud instance is required or assumed.
- The preflight requires at least **14 GiB total VRAM and 12 GiB free VRAM**. If the free runtime does not meet this, the notebook stops before training.
- Free Colab GPU availability is not guaranteed. Reconnect later or use another free CUDA runtime if capacity is unavailable.
- The notebook can resume the previous adapter from `MyDrive/VELNOR-training-runs/qwen3-8b-20261112T123354Z/outputs/adapter` if it exists. Otherwise it starts a fresh adapter.

The previous run's validation loss is not used as a threshold because this run has a new held-out set. The resumed adapter is first evaluated on the current 30-example validation set, and only then are continuation checkpoints compared against that baseline.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU attached. Select an available free GPU runtime; CPU training is disabled.')
props = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info(0)
free_gib, total_gib = free / 1024**3, total / 1024**3
print('GPU:', props.name, '| total GiB:', round(total_gib, 2), '| free GiB:', round(free_gib, 2))
if total_gib < 14 or free_gib < 12:
    raise RuntimeError('Qwen3-8B QLoRA needs at least 14 GiB total and 12 GiB free. No training started.')

In [ ]:
REPO = '/content/velnor-general-qwen3'
!git clone --depth 1 --branch training/general-purpose-curriculum-and-eval https://github.com/vectolith/velnor.git $REPO
%cd /content/velnor-general-qwen3

In [ ]:
!python -m pip install -q -r training/requirements-qwen3-qlora.txt
import torch
assert torch.cuda.is_available(), 'CUDA was lost during dependency installation.'
print('CUDA:', torch.version.cuda, '| PyTorch:', torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess
RUNS = Path('/content/drive/MyDrive/VELNOR-training-runs')
RUNS.mkdir(parents=True, exist_ok=True)
PRIOR = RUNS / 'qwen3-8b-20261112T123354Z' / 'outputs' / 'adapter'
RESUME = PRIOR if (PRIOR / 'adapter_config.json').is_file() else None
run_name = 'qwen3-8b-general-' + subprocess.check_output(['date','-u','+%Y%m%dT%H%M%SZ'], text=True).strip()
RUN_DIR = RUNS / run_name
OUT_DIR = RUN_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=False)
os.environ['VELNOR_QWEN3_OUTPUT'] = str(OUT_DIR)
os.environ['VELNOR_QWEN3_MAX_SEQ_LEN'] = '256'
os.environ['VELNOR_QWEN3_EPOCHS'] = '3'
os.environ['VELNOR_QWEN3_GRAD_ACCUM'] = '8'
if RESUME:
    os.environ['VELNOR_QWEN3_RESUME'] = str(RESUME)
    print('Resuming previous adapter:', RESUME)
else:
    os.environ.pop('VELNOR_QWEN3_RESUME', None)
    print('Previous adapter not found; starting a fresh QLoRA adapter.')
print('Run output:', OUT_DIR)

In [ ]:
%%bash
set -euo pipefail
python -u training/train_qwen3_8b_qlora.py 2>&1 | tee "$VELNOR_QWEN3_OUTPUT/training.log"
test -s "$VELNOR_QWEN3_OUTPUT/adapter/adapter_config.json"
test -s "$VELNOR_QWEN3_OUTPUT/training_report.json"
echo 'QLoRA training and artifact checks completed.'

In [ ]:
import json
report_path = OUT_DIR / 'training_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('Status:', report['status'])
print('Base model:', report['base_model'])
print('Dataset:', report['dataset'])
print('Base rubric:', report['rubric']['base']['passed'], '/', report['rubric']['base']['total'])
print('Tuned rubric:', report['rubric']['tuned']['passed'], '/', report['rubric']['tuned']['total'])
print('Rubric delta:', report['rubric']['delta_passed'])
print('Validation:', report['validation'])

In [ ]:
import shutil
archive = shutil.make_archive(str(RUN_DIR / 'velnor-qwen3-8b-general-artifacts'), 'zip', root_dir=OUT_DIR)
print('Adapter, training report, and log archived at:', archive)
print('All run artifacts are already stored in Google Drive:', RUN_DIR)

In [ ]:
from google.colab import files
files.download(str(RUN_DIR / 'velnor-qwen3-8b-general-artifacts.zip'))